## 1. Base de dados textuais

**Justificativa e explicação da escolha:** Escolhemos o Project Gutenberg por reunir várias obras de domínio público. A estrutura dos arquivos disponibilizados permite a raspagem automatizada e o processamento completo de textos para tarefas de PLN.

**Adequação dos dados às tarefas de PLN (volume, abrangência, variedade):** As obras foram obtidas diretamente via rota de harvest do Gutenberg (`/robot/harvest?filetypes[]=html`), extraindo arquivos HTML compactados (.zip) em inglês e identificados pelo ID do livro. O script descompacta os arquivos, extrai o texto limpo, amostra trechos do início (20%), meio (50%) e fim (80%) de cada livro e armazena os dados processados em um arquivo CSV.

**Limitação:** A amostragem baseia-se na captura automatizada da página de harvest de arquivos HTML, sem raspagem dedicada das páginas de metadados individuais (como assuntos detalhados ou classificação da Library of Congress). Além disso, a limpeza do texto depende da presença das marcações de cabeçalho/rodapé padrão do Gutenberg.

**Organização e interpretabilidade da base de dados (dicionário de dados):**

| Coluna | Tipo | Descrição |
|---|---|---|
| `book_id` | texto | Identificador único do livro no Project Gutenberg |
| `file_path` | texto | Caminho local do arquivo HTML extraído |
| `trecho_1` / `trecho_2` / `trecho_3` | texto | Trechos brutos de 200 palavras extraídos a 20%, 50% e 80% do texto |
| `trecho_1_clean` / `trecho_2_clean` / `trecho_3_clean` | texto | Trechos de 200 palavras pré-processados (minúsculas, sem stopwords e pontuação) |
| `tokens_clean` | lista | Lista de tokens filtrados do livro completo (sem stopwords nem pontuação) |
| `n_tokens` | inteiro | Quantidade total de tokens brutos do livro |
| `n_tokens_clean` | inteiro | Quantidade total de tokens limpos do livro |

## 2. Script Python — scraping, acesso e extração dos dados

**Funcionamento e reprodutibilidade:** O script realiza o scraping da página de harvest oficial do Gutenberg, baixa os arquivos `.zip` contendo os HTMLs dos livros e executa o tratamento textual (remoção de tags HTML, licenças e normalização). O fluxo é reprodutível e otimizado: antes de efetuar o download, o script verifica se os arquivos já existem localmente para evitar requisições redundantes.

**Pontos de atenção identificados:** Utilização de um `User-Agent` identificável (`AcademicResearchBot/1.0`), supressão pontual de avisos SSL para mirrors do Gutenberg (`verify=False`) e inclusão de pausa de 2 segundos entre requisições (`time.sleep(2)`) para respeitar os termos do `robot_access.html`.

**Documentação dos procedimentos realizados:** O pipeline é estruturado nas seguintes etapas:
1. Raspagem de links HTML/ZIP via Gutenberg Harvest (`/robot/harvest`) e extração do ID dos livros via Regex.
2. Download e sincronização local dos arquivos `.zip` com verificação de cache em disco.
3. Descompactação dos `.zip` e parsing do HTML com `BeautifulSoup` e `Regex` (remoção de tags HTML, scripts e licenças padrão do Gutenberg).
4. Amostragem de 3 trechos por livro (início, meio e fim) e pré-processamento via NLTK (tokenização, conversão para minúsculas, remoção de stopwords e pontuação).
5. Cálculo de métricas de contagem de tokens (`n_tokens`, `n_tokens_clean`) e exportação consolidada para o arquivo CSV (`gutenberg_books.csv`).

In [1]:
import glob
import os
import re
import time
from urllib.parse import urljoin
from bs4 import BeautifulSoup
import pandas as pd
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

output_folder = "gutenberg_harvest"
os.makedirs(output_folder, exist_ok=True)

HARVEST_URL = "https://www.gutenberg.org/robot/harvest?filetypes[]=html"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AcademicResearchBot/1.0"
}

MAX_BOOKS = 20
downloaded_count = 0

print(f"Iniciando raspagem oficial via Gutenberg Harvest: {HARVEST_URL}")
print("Respeitando intervalo de 2 segundos (-w 2) conforme robot_access.html...\n")

current_harvest_url = HARVEST_URL

while current_harvest_url and downloaded_count < MAX_BOOKS:
    try:
        response = requests.get(
            current_harvest_url, headers=headers, timeout=15, verify=False
        )
        if response.status_code != 200:
            print(
                f"Erro ao acessar harvest ({response.status_code}). Interrompendo..."
            )
            break

        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.find_all("a", href=True)

        file_links = []
        for l in links:
            href = l["href"]
            if (
                "aleph.gutenberg.org" in href
                or href.endswith(".zip")
                or "-h.zip" in href
            ):
                file_links.append(urljoin(current_harvest_url, href))

        for zip_url in file_links:
            if downloaded_count >= MAX_BOOKS:
                break

            # Extrai o ID apenas a partir do nome do arquivo final (ex: 1342-h.zip -> 1342)
            filename = os.path.basename(zip_url)
            match = re.search(r"(\d+)", filename)
            book_id = match.group(1) if match else f"book_{downloaded_count}"
            save_path = os.path.join(output_folder, f"{book_id}-h.zip")

            if os.path.exists(save_path):
                print(f"Livro ID {book_id} já existe localmente. Pulando...")
                continue

            time.sleep(2)
            try:
                r_file = requests.get(
                    zip_url, headers=headers, timeout=20, verify=False
                )
                if r_file.status_code == 200 and len(r_file.content) > 1000:
                    with open(save_path, "wb") as f:
                        f.write(r_file.content)
                    downloaded_count += 1
                    print(
                        f"[{downloaded_count}/{MAX_BOOKS}] Baixado com sucesso: {book_id}"
                    )
            except Exception as e_file:
                print(f"Erro no download de {zip_url}: {e_file}")

        next_page = soup.find("a", string=re.compile(r"Next\s*Page", re.I))
        if next_page and "href" in next_page.attrs:
            current_harvest_url = urljoin(HARVEST_URL, next_page["href"])
            time.sleep(2)
        else:
            current_harvest_url = None

    except Exception as e:
        print(f"Falha na execução do harvest: {e}")
        break

print(
    f"\nColeta concluída! {downloaded_count} arquivos baixados em '{output_folder}'."
)

Iniciando raspagem oficial via Gutenberg Harvest: https://www.gutenberg.org/robot/harvest?filetypes[]=html
Respeitando intervalo de 2 segundos (-w 2) conforme robot_access.html...

Livro ID 10084 já existe localmente. Pulando...
Livro ID 1554 já existe localmente. Pulando...
Livro ID 1680 já existe localmente. Pulando...
Livro ID 71 já existe localmente. Pulando...
Livro ID 1957 já existe localmente. Pulando...
Livro ID 1940 já existe localmente. Pulando...
Livro ID 1837 já existe localmente. Pulando...
Livro ID 1749 já existe localmente. Pulando...
Livro ID 245 já existe localmente. Pulando...
Livro ID 1729 já existe localmente. Pulando...
Livro ID 1925 já existe localmente. Pulando...
Livro ID 2452 já existe localmente. Pulando...
Livro ID 1715 já existe localmente. Pulando...
Livro ID 1649 já existe localmente. Pulando...
Livro ID 535 já existe localmente. Pulando...
Livro ID 2303 já existe localmente. Pulando...
Livro ID 2304 já existe localmente. Pulando...
Livro ID 430 já existe 

## 3. Limpeza e preparação dos dados

**Dados foram tokenizados?** Sim, foi utilizado o `nltk.tokenize.word_tokenize` em duas frentes: para gerar a lista global de palavras do livro na coluna `tokens_clean` e para processar individualmente os três trechos amostrados (`trecho_1_clean`, `trecho_2_clean`, `trecho_3_clean`).

**Dados foram normalizados?** Sim, todo o texto é convertido para minúsculas (`.lower()`) antes da tokenização. Além disso, a função de extração colapsa múltiplos espaços em branco, tabulações e quebras de linha em um único espaço via expressão regular (`re.sub(r"\s+", " ", text)`).

**Remoção de ruídos:** Sim. O `BeautifulSoup` descarta tags HTML e elementos estruturais (`script`, `style`, `header`, `footer`). Em seguida, expressões regulares localizam e removem os blocos de licença jurídica e cabeçalhos padrão (`*** START OF... ***` e `*** END OF... ***`). Arquivos com texto limpo menor que 1000 caracteres são descartados.

**Remoção de stopwords e pontuação:** Após a tokenização, os tokens são filtrados contra a lista de stopwords em inglês do NLTK (`stopwords.words("english")`) e os caracteres de pontuação do módulo `string.punctuation`.

In [2]:
import glob
import os
import re
import string
import zipfile
from bs4 import BeautifulSoup
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def extract_text_from_html(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html_content = f.read()

    # Tenta utilizar o parser lxml (muito mais rápido), caindo para html.parser se não instalado
    try:
        soup = BeautifulSoup(html_content, "lxml")
    except Exception:
        soup = BeautifulSoup(html_content, "html.parser")

    for script in soup(["script", "style", "header", "footer"]):
        script.extract()

    text = soup.get_text(separator=" ")

    start_match = re.search(
        r"\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        re.IGNORECASE,
    )
    end_match = re.search(
        r"\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK.*?\*\*\*",
        text,
        re.IGNORECASE,
    )

    if start_match and end_match:
        text = text[start_match.end() : end_match.start()]
    elif start_match:
        text = text[start_match.end() :]

    return re.sub(r"\s+", " ", text).strip()


def extrair_trechos(texto, qtd_trechos=3, min_palavras=200):
    palavras = texto.split()
    total = len(palavras)
    if total < (qtd_trechos * min_palavras):
        return [" ".join(palavras)] * qtd_trechos

    marcadores = [int(total * 0.2), int(total * 0.5), int(total * 0.8)]
    return [" ".join(palavras[idx : idx + min_palavras]) for idx in marcadores]


def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = word_tokenize(text.lower(), language="english")
    tokens_filtered = [
        tok
        for tok in tokens
        if tok not in STOPWORDS_EN and not all(ch in PUNCT for ch in tok)
    ]
    return " ".join(tokens_filtered)


# Descompactação dos arquivos baixados
output_folder = "gutenberg_harvest"
zip_files = glob.glob(os.path.join(output_folder, "*.zip"))
for zip_path in zip_files:
    try:
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(output_folder)
    except Exception as e:
        print(f"Erro ao descompactar {zip_path}: {e}")

# Processamento dos HTMLs
html_files = glob.glob(
    os.path.join(output_folder, "**/*.htm*"), recursive=True
)
records = []

for file_path in html_files:
    match = re.search(r"(\d+)", os.path.basename(file_path))
    if not match:
        continue

    b_id = match.group(1)
    cleaned_text = extract_text_from_html(file_path)
    if len(cleaned_text) < 500:
        continue

    trechos = extrair_trechos(cleaned_text, qtd_trechos=3, min_palavras=200)

    records.append(
        {
            "book_id": str(b_id),
            "file_path": file_path,
            "raw_text": cleaned_text,
            "text": cleaned_text,  # Mantém a chave 'text' para evitar KeyError
            "trecho_1": trechos[0],
            "trecho_2": trechos[1],
            "trecho_3": trechos[2],
        }
    )

df_books = pd.DataFrame(records)
print(f"Livros válidos processados: {len(df_books)}")

Erro ao descompactar gutenberg_harvest\245-h.zip: [Errno 22] Invalid argument: 'gutenberg_harvest\\245-h\\images\\139.jpg'
Erro ao descompactar gutenberg_harvest\59828-h.zip: [Errno 22] Invalid argument: 'gutenberg_harvest\\59828-h\\images\\fig060.png'
Livros válidos processados: 179


In [3]:
import glob
import os
import re
import string
import zipfile
from bs4 import BeautifulSoup
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Garante o download dos recursos do NLTK
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    tokens = word_tokenize(text.lower(), language="english")
    tokens_filtered = [
        tok
        for tok in tokens
        if tok not in STOPWORDS_EN and not all(ch in PUNCT for ch in tok)
    ]
    return " ".join(tokens_filtered)


def build_dataset_from_harvest(base_folder):
    zip_files = glob.glob(os.path.join(base_folder, "*.zip"))
    print(f"Descompactando {len(zip_files)} arquivos .zip...")
    for zip_path in zip_files:
        try:
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(base_folder)
        except Exception as e:
            print(f"Erro ao descompactar {zip_path}: {e}")

    html_files = glob.glob(
        os.path.join(base_folder, "**/*.htm*"), recursive=True
    )

    data = []
    print(f"Total de arquivos HTML encontrados: {len(html_files)}")

    for file_path in html_files:
        filename = os.path.basename(file_path)
        match = re.search(r"(\d+)", filename)
        if not match:
            continue

        book_id = match.group(1)
        cleaned_text = extract_text_from_html(file_path)

        if len(cleaned_text) < 1000:
            continue

        trechos = extrair_trechos(cleaned_text, qtd_trechos=3, min_palavras=200)

        data.append(
            {
                "book_id": book_id,
                "file_path": file_path,
                "text": cleaned_text,
                "trecho_1": trechos[0],
                "trecho_2": trechos[1],
                "trecho_3": trechos[2],
            }
        )

    df = pd.DataFrame(data)
    return df


# 1. Carrega o dataset
df_books = build_dataset_from_harvest("gutenberg_harvest")
print(f"Livros válidos processados: {len(df_books)}")

print("Extraindo e limpando os trechos...")

# 2. Processamento PLN dos trechos
for col in ["trecho_1", "trecho_2", "trecho_3"]:
    df_books[f"{col}_clean"] = df_books[col].apply(preprocess_text)

# 3. Tokenização global
df_books["tokens"] = df_books["text"].apply(
    lambda t: word_tokenize(t.lower(), language="english")
)
df_books["tokens_clean"] = df_books["tokens"].apply(
    lambda toks: [
        t
        for t in toks
        if t not in STOPWORDS_EN and not all(c in PUNCT for c in t)
    ]
)

# 4. Limpeza de colunas pesadas do lote atual
df_novos_dados = df_books.drop(
    columns=["raw_text", "text", "tokens"], errors="ignore"
)

csv_saida = "gutenberg_books.csv"

# 5. Atualização acumulativa e correta no CSV base
if os.path.exists(csv_saida):
  df_base = pd.read_csv(csv_saida)
else:
  df_base = pd.DataFrame()

if not df_base.empty and not df_novos_dados.empty:
  df_base["book_id"] = df_base["book_id"].astype(str)
  df_novos_dados["book_id"] = df_novos_dados["book_id"].astype(str)

  # Concatena a base existente com os novos dados e remove duplicatas (mantém a versão mais recente)
  df_final = pd.concat([df_base, df_novos_dados]).drop_duplicates(
      subset=["book_id"], keep="last"
  )
else:
  df_final = df_novos_dados if not df_novos_dados.empty else df_base

# 6. Salva o resultado final preservando toda a base
df_final.to_csv(csv_saida, index=False, encoding="utf-8-sig")

Descompactando 167 arquivos .zip...
Total de arquivos HTML encontrados: 179
Livros válidos processados: 179
Extraindo e limpando os trechos...


In [4]:
import nltk
import string

for pkg in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Aviso: não foi possível baixar {pkg}: {e}")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

STOPWORDS_EN = set(stopwords.words("english"))
PUNCT = set(string.punctuation)


def tokenize_text(text):
    """Normaliza (minúsculas) e tokeniza o texto usando o NLTK."""
    return word_tokenize(text.lower(), language="english")


def remove_stopwords_punct(tokens):
    """Remove stopwords e pontuação da lista de tokens."""
    return [
        tok for tok in tokens
        if tok not in STOPWORDS_EN
        and not all(ch in PUNCT for ch in tok)
    ]


# Célula 4: Contagem e inspeção dos tokens já processados na Célula 3

# Calcula a quantidade de tokens brutos e limpos
df_books["n_tokens"] = df_books["tokens"].apply(len)
df_books["n_tokens_clean"] = df_books["tokens_clean"].apply(len)

# Visualiza as primeiras linhas com as contagens
df_books[
    ["book_id", "n_tokens", "n_tokens_clean", "tokens_clean"]
].head()

,book_id,n_tokens,n_tokens_clean,tokens_clean
0,11,35204,15443,"[alice, ’, adventures, wonderland, project, gu..."
1,1260,229840,97850,"[jane, eyre, project, gutenberg, jane, eyre, a..."
2,1342,152928,63239,"[pride, prejudice, project, gutenberg, preface..."
3,145,377168,165954,"[middlemarch, project, gutenberg, middlemarch,..."
4,2363,26078,10380,"[incognita, love, duty, reconcil, ’, novel, wi..."


## 4. Bonus round

**Stemming e lematização:** não implementado nesta versão.

**Bases adicionais (comparação entre fontes/recortes):** não implementado nesta versão.

**Automatização na coleta:** parcialmente implementado, o script de download verifica arquivos já existentes e evita baixá-los de novo, mas não há agendamento para atualização periódica automática.